# BookToText: Vietnamese History PDF OCR Pipeline

Self-contained Colab notebook. Downloads Vietnamese history PDFs, runs OCR with **PaddleOCR + VietOCR**, saves per-page YAML output, and pushes to GitHub.

**Runtime:** GPU (T4 or better)

## 1. GPU Check

In [ ]:

import subprocess, sys
print('=== CUDA Version ===')
!nvcc --version 2>/dev/null || echo 'nvcc not found'
print('=== GPU Info ===')
!nvidia-smi 2>/dev/null || echo 'nvidia-smi not found'
print(f'=== Python: {sys.version} ===')

## 2. Install System Dependencies

In [ ]:

!apt-get update -qq && apt-get install -y -qq poppler-utils > /dev/null 2>&1
!echo "poppler-utils installed"
!pdftoppm -v 2>&1 | head -1

## 3. Install PaddlePaddle GPU

In [ ]:
!pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
# Fix NCCL conflict: paddlepaddle-gpu pins nvidia-nccl-cu12==2.25.1
# but torch 2.11.0+cu128 needs 2.28.9 (for ncclCommShrink symbol)
# Must restart runtime after this cell for fix to take effect!
!pip install --force-reinstall nvidia-nccl-cu12==2.28.9 -q

## 4. Install Python Packages

In [ ]:
!pip install paddleocr 'paddlex[ocr]' vietocr pdf2image pypdf pyyaml tqdm click opencv-python-headless boto3 -q
!pip install --upgrade pillow==10.2.0

In [ ]:
from paddleocr import TextDetection, LayoutDetection

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from PIL import Image as PILImage
import numpy as np, cv2, yaml, unicodedata, os, re, logging, time, glob
from io import BytesIO
from pathlib import Path
from tqdm.auto import tqdm
print('All imports OK')

## 5. Download Books

In [ ]:
!mkdir -p /kaggle/working/input /kaggle/working/output
!wget -q --show-progress -O /tmp/vietnam_history_pdf.zip https://pub-3d5ab760174a48fda9acb3630968631e.r2.dev/vietnam_history_pdf.zip
!unzip -o -q /tmp/vietnam_history_pdf.zip -d /kaggle/working/input/

pdfs = [f for f in os.listdir('/kaggle/working/input') if f.endswith('.pdf')]
print(f'Found {len(pdfs)} PDF files:')
for pdf in sorted(pdfs):
    size_mb = os.path.getsize(f'/kaggle/working/input/{pdf}') / (1024*1024)
    print(f'  {pdf} ({size_mb:.1f} MB)')

## 6. Define Pipeline Code

In [ ]:
# === Logger & FileSystem ===
def get_logger(name):
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    return logger

class FileInput:
    def __init__(self, name: str, data: bytes):
        self.name = name; self.data = data

class FileSystem:
    def __init__(self, base: str): self.base = base
    def put(self, inp):
        p = Path(self.base) / inp.name
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(inp.data)
    def ls(self, prefix=None):
        base = Path(self.base)
        if not base.exists(): return []
        files = [str(p.relative_to(base)) for p in base.rglob('*') if p.is_file()]
        if prefix: files = [f for f in files if f.startswith(prefix)]
        return files
    def read(self, name):
        return (Path(self.base) / name).read_bytes()

print('Logger & FileSystem OK')

In [ ]:
# === Data Classes ===
class Image:
    def __init__(self, book_name, page_name, data):
        self.book_name = book_name; self.page_name = page_name; self.data = data
    def get_book_name(self): return self.book_name
    def get_page_name(self): return self.page_name
    def get_data(self):
        arr = np.frombuffer(self.data, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is None: raise ValueError(f'Failed to decode {self.page_name}')
        return img

class LayoutElement:
    def __init__(self, label, score, bbox):
        self.label = label; self.score = score; self.bbox = bbox

class TextElement:
    def __init__(self, text, confidence, box, label='text'):
        self.text = text; self.confidence = confidence; self.box = box; self.label = label

class DetectedText:
    def __init__(self, box, score):
        self.box = box; self.score = score

class Table:
    def __init__(self, bbox, html, score):
        self.bbox = bbox; self.html = html; self.score = score
    def to_dict(self):
        return {'bbox': self.bbox, 'html': self.html, 'score': self.score}

class Page:
    def __init__(self, doc_name, number, header, content, footer, tables=None):
        self.doc_name = doc_name; self.number = number; self.header = header
        self.content = content; self.footer = footer; self.tables = tables or []
    def to_dict(self):
        return {'doc_name': self.doc_name, 'number': self.number, 'header': self.header,
                'content': self.content, 'footer': self.footer,
                'tables': [t.to_dict() for t in self.tables]}

print('Data Classes OK')

In [ ]:
# === PDF & Converter ===
from pdf2image import convert_from_bytes, pdfinfo_from_bytes

CHUNK_SIZE = 50  # Convert PDF pages in batches to avoid OOM

class PDF:
    def __init__(self, name, data):
        self._name = name; self._data = data
    def get_name(self): return self._name
    def get_data(self): return self._data
    def get_page_count(self):
        return int(pdfinfo_from_bytes(self._data)['Pages'])

class PyPDFConverter:
    def __init__(self, dpi=300):
        self._dpi = dpi; self._logger = get_logger(self.__class__.__name__)
    def convert(self, pdf, start_page=0, end_page=None, dest=None):
        total_pages = pdf.get_page_count()
        last = end_page if end_page is not None else total_pages - 1
        book = unicodedata.normalize('NFC', pdf.get_name())

        # Determine which pages are missing
        missing = None
        if dest is not None:
            img_dir = f'images/{book}'
            existing = set()
            for item in dest.ls():
                n = unicodedata.normalize('NFC', item)
                if n.startswith(img_dir + '/') and n.endswith('.png'):
                    try: existing.add(int(n.split('/')[-1].replace('.png','')))
                    except: pass
            missing = set(range(start_page, last+1)) - existing
            if not missing:
                self._logger.info(f'Skipping {book}: all pages already converted')
                return []
            self._logger.info(f'{book}: {len(missing)} pages to convert')
        else:
            missing = set(range(start_page, last+1))

        # Convert in chunks with progress bar
        images = []
        pages_sorted = sorted(missing)
        chunks = [pages_sorted[i:i+CHUNK_SIZE] for i in range(0, len(pages_sorted), CHUNK_SIZE)]

        with tqdm(total=len(pages_sorted), desc=f'PDF→IMG {book[:30]}', unit='pg') as pbar:
            for chunk in chunks:
                chunk_start = chunk[0]
                chunk_end = chunk[-1]
                pil_images = convert_from_bytes(
                    pdf.get_data(), dpi=self._dpi,
                    first_page=chunk_start+1,  # pdf2image is 1-indexed
                    last_page=chunk_end+1
                )
                for i, pil in enumerate(pil_images):
                    actual = chunk_start + i
                    if actual not in missing: continue
                    buf = BytesIO(); pil.save(buf, format='PNG'); buf.seek(0)
                    images.append(Image(pdf.get_name(), f'page_{actual}', buf.getvalue()))
                    pbar.update(1)
                # Free memory between chunks
                del pil_images
                import gc; gc.collect()

        self._logger.info(f'{book}: converted {len(images)} pages')
        return images

print('PDF & Converter OK (chunked, with progress)')

In [ ]:
# === Box Merging & Geometry Utils ===
def _merge_boxes_to_lines(boxes, y_overlap_threshold=0.5, x_gap_threshold=50):
    if not boxes: return []
    sorted_boxes = sorted(boxes, key=lambda b: (min(p[1] for p in b), min(p[0] for p in b)))
    lines = []
    for box in sorted_boxes:
        x1 = min(p[0] for p in box); y1 = min(p[1] for p in box)
        x2 = max(p[0] for p in box); y2 = max(p[1] for p in box)
        placed = False
        for line in lines:
            lb = line[-1]
            ly1 = min(p[1] for p in lb); ly2 = max(p[1] for p in lb)
            lh = ly2 - ly1; bh = y2 - y1; mh = min(bh, lh)
            if mh <= 0: continue
            y_overlap = max(0, min(y2, ly2) - max(y1, ly1))
            if y_overlap / mh >= y_overlap_threshold:
                lx2 = max(p[0] for p in lb)
                if x1 - lx2 <= x_gap_threshold:
                    line.append(box); placed = True; break
        if not placed: lines.append([box])
    result = []
    for line in lines:
        line.sort(key=lambda b: min(p[0] for p in b))
        min_x = min(min(p[0] for p in b) for b in line)
        min_y = min(min(p[1] for p in b) for b in line)
        max_x = max(max(p[0] for p in b) for b in line)
        max_y = max(max(p[1] for p in b) for b in line)
        result.append([[min_x, min_y], [max_x, max_y]])
    return result

def _bbox_overlap_ratio(text_quad, layout_bbox):
    xs = [p[0] for p in text_quad]; ys = [p[1] for p in text_quad]
    tx1, tx2 = min(xs), max(xs); ty1, ty2 = min(ys), max(ys)
    lx1, ly1, lx2, ly2 = layout_bbox
    ix1, iy1 = max(tx1, lx1), max(ty1, ly1)
    ix2, iy2 = min(tx2, lx2), min(ty2, ly2)
    if ix1 >= ix2 or iy1 >= iy2: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area = (tx2 - tx1) * (ty2 - ty1)
    return inter / area if area else 0.0

# === Page Number Parsing ===
_ROMAN_PATTERN = re.compile(r'^[ivxlcdmIVXLCDM]+$')
def _is_valid_roman(s):
    return bool(re.match(r'^M*(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$', s, re.IGNORECASE))
def _roman_to_int(s):
    if not _is_valid_roman(s): return 0
    vals = {'i':1,'v':5,'x':10,'l':50,'c':100,'d':500,'m':1000}
    total, prev = 0, 0
    for c in reversed(s.lower()):
        v = vals.get(c,0); total += -v if v < prev else v; prev = v
    return total
def _parse_page_number(s):
    if not s: return 0
    s = s.strip()
    if not s: return 0
    if s.isdigit(): return int(s)
    m = re.search(r'\d+', s)
    if m: return int(m.group())
    if _ROMAN_PATTERN.match(s) and _is_valid_roman(s): return _roman_to_int(s)
    return 0

print('Geometry & Parsing Utils OK')

In [ ]:
# === OCR Engine: PaddleVietOCR ===
import threading
_MODEL_CACHE = {}
_MODEL_CACHE_LOCK = threading.Lock()

def _detect_device(device_id: int = 0):
    """Returns (paddle_device, torch_device). PaddleX='gpu:N', torch='cuda:N'."""
    env = os.environ.get('BOOKTOTEXT_DEVICE','').strip().lower()
    if env == 'cpu': return 'cpu', 'cpu'
    if env in ('cuda', 'gpu'): return f'gpu:{device_id}', f'cuda:{device_id}'
    if env == 'mps': return 'cpu', 'mps'
    try:
        import torch
        if torch.cuda.is_available(): return f'gpu:{device_id}', f'cuda:{device_id}'
        if hasattr(torch.backends,'mps') and torch.backends.mps.is_available(): return 'cpu', 'mps'
    except: pass
    return 'cpu', 'cpu'

class PaddleVietOCR:
    LABEL_MAP = {'text':'text','title':'page_title','table':'table',
                 'figure':'figure','header':'header','footer':'footer',
                 'page_number':'number','footnote':'footnote'}

    def __init__(self, device_id=0, debug=False, padding=4):
        self._debug = debug; self._padding = padding
        self._device_id = device_id
        self._log = get_logger(f'OCR-GPU{device_id}')
        paddle_device, torch_device = _detect_device(device_id)
        self._paddle_device = paddle_device
        self._torch_device = torch_device

        # Timing stats (thread-safe)
        self._stats = {'det': [], 'layout': [], 'rec': [], 'total': [], 'pages': 0}
        self._stats_lock = threading.Lock()

        # Text detection (server model for higher accuracy)
        self._log.info(f'Loading TextDetection model PP-OCRv5_server_det (device={paddle_device})...')
        t0 = time.time()
        cache_key = f'textdet_{paddle_device}'
        with _MODEL_CACHE_LOCK:
            if cache_key not in _MODEL_CACHE:
                _MODEL_CACHE[cache_key] = TextDetection(
                    model_name='PP-OCRv5_server_det', device=paddle_device)
            self._detector = _MODEL_CACHE[cache_key]
        self._log.info(f'TextDetection ready ({time.time()-t0:.1f}s)')

        # Layout detection
        self._log.info(f'Loading LayoutDetection model (device={paddle_device})...')
        t0 = time.time()
        lkey = f'layout_{paddle_device}'
        with _MODEL_CACHE_LOCK:
            if lkey not in _MODEL_CACHE:
                _MODEL_CACHE[lkey] = LayoutDetection(device=paddle_device)
            self._layout = _MODEL_CACHE[lkey]
        self._log.info(f'LayoutDetection ready ({time.time()-t0:.1f}s)')

        # VietOCR recognition
        self._log.info(f'Loading VietOCR model (device={torch_device})...')
        t0 = time.time()
        config = Cfg.load_config_from_name('vgg_transformer')
        config['cnn']['pretrained'] = True
        config['predictor']['beamsearch'] = True
        config['device'] = torch_device
        vkey = f'vietocr_{torch_device}'
        with _MODEL_CACHE_LOCK:
            if vkey not in _MODEL_CACHE:
                _MODEL_CACHE[vkey] = Predictor(config)
            self._recognizer = _MODEL_CACHE[vkey]
        self._log.info(f'VietOCR ready ({time.time()-t0:.1f}s)')

        # Table recognition (lazy init)
        self._table_rec = None
        print(f'OCR Engine ready (GPU{device_id}): Paddle={paddle_device}, VietOCR={torch_device}')

    def print_stats(self):
        n = self._stats['pages']
        if n == 0:
            print('No pages processed yet')
            return
        avg_det = sum(self._stats['det']) / n
        avg_lay = sum(self._stats['layout']) / n
        avg_rec = sum(self._stats['rec']) / n
        avg_total = sum(self._stats['total']) / n
        print(f'--- OCR Timing Stats ({n} pages) ---')
        print(f'   PaddleOCR TextDetection:   avg {avg_det*1000:.0f}ms/page')
        print(f'   PaddleOCR LayoutDetection: avg {avg_lay*1000:.0f}ms/page')
        print(f'   VietOCR Recognition:       avg {avg_rec*1000:.0f}ms/page')
        print(f'   Total per page:            avg {avg_total:.2f}s/page')
        print(f'   Throughput:                {n / sum(self._stats["total"]):.1f} pages/s')

    def ocr(self, image):
        img = image.get_data()
        t0 = time.time()

        boxes = self._detect_text_boxes(image)
        t1 = time.time()

        layouts = self._detect_layout(image)
        t2 = time.time()

        texts = self._recognize_texts(img, boxes)
        t3 = time.time()

        self._assign_labels(texts, layouts)
        tables = self._process_tables(img, layouts)
        t4 = time.time()

        # Record stats (thread-safe)
        with self._stats_lock:
            self._stats['det'].append(t1 - t0)
            self._stats['layout'].append(t2 - t1)
            self._stats['rec'].append(t3 - t2)
            self._stats['total'].append(t4 - t0)
            self._stats['pages'] += 1
            n = self._stats['pages']

        # Log every 10 pages
        if n % 10 == 0:
            avg_det = sum(self._stats['det'][-10:]) / 10
            avg_lay = sum(self._stats['layout'][-10:]) / 10
            avg_rec = sum(self._stats['rec'][-10:]) / 10
            print(f'  [GPU{self._device_id} pg {n}] det={avg_det*1000:.0f}ms lay={avg_lay*1000:.0f}ms rec={avg_rec*1000:.0f}ms ({len(boxes)} boxes) total={t4-t0:.2f}s')

        return self._build_page(image.get_book_name(), texts, tables)

    def _detect_text_boxes(self, image):
        out = self._detector.predict(image.get_data(), batch_size=1)
        if not out: return []
        res = out[0]
        if self._debug:
            os.makedirs(f'./debug/{image.get_book_name()}/text', exist_ok=True)
            res.save_to_img(f'./debug/{image.get_book_name()}/text/{image.get_page_name()}.png')
        data = res.json['res']
        quads = data.get('dt_polys', [])
        scores = data.get('dt_scores', [])
        rects = []
        for quad in quads:
            x1 = int(min(p[0] for p in quad)); y1 = int(min(p[1] for p in quad))
            x2 = int(max(p[0] for p in quad)); y2 = int(max(p[1] for p in quad))
            rects.append([[x1,y1],[x2,y2]])
        merged = _merge_boxes_to_lines(rects)
        detected = []
        for i, box in enumerate(merged):
            score = max(scores) if scores else 1.0
            detected.append(DetectedText(box, float(score)))
        return detected

    def _detect_layout(self, image):
        out = self._layout.predict(image.get_data())
        if not out: return []
        res = out[0]
        if self._debug:
            os.makedirs(f'./debug/{image.get_book_name()}/layout', exist_ok=True)
            res.save_to_img(f'./debug/{image.get_book_name()}/layout/{image.get_page_name()}.png')
        regions = []
        for box in res.get('boxes', []) or []:
            label = self.LABEL_MAP.get(str(box.get('label','text')).lower(), 'text')
            regions.append(LayoutElement(label, float(box.get('score',0.0)),
                                         list(box.get('coordinate',[0,0,0,0]))))
        return regions

    def _recognize_texts(self, img, detected):
        elements = []
        for det in detected:
            xs = [p[0] for p in det.box]; ys = [p[1] for p in det.box]
            x1 = max(0, int(min(xs)) - self._padding)
            y1 = max(0, int(min(ys)) - self._padding)
            x2 = min(img.shape[1], int(max(xs)) + self._padding)
            y2 = min(img.shape[0], int(max(ys)) + self._padding)
            cropped = img[y1:y2, x1:x2]
            if cropped.size == 0: continue
            try:
                pil = PILImage.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
                text = self._recognizer.predict(pil)
                elements.append(TextElement(text, det.score, det.box, 'text'))
            except Exception as e:
                self._log.warning(f'Recognition failed: {e}')
        return elements

    def _assign_labels(self, texts, layouts):
        if not layouts: return
        sorted_layouts = sorted(layouts, key=lambda l: -l.score)
        for elem in texts:
            best_label, best_overlap = 'text', 0.0
            for layout in sorted_layouts:
                overlap = _bbox_overlap_ratio(elem.box, layout.bbox)
                if overlap > best_overlap:
                    best_overlap = overlap; best_label = layout.label
            if best_overlap >= 0.5: elem.label = best_label

    def _process_tables(self, img, layouts):
        tables = []
        for layout in layouts:
            if layout.label != 'table': continue
            with self._stats_lock:  # reuse lock for lazy init
                if self._table_rec is None:
                    self._log.info('Loading TableRecognitionPipelineV2 (first table)...')
                    from paddleocr import TableRecognitionPipelineV2
                    self._table_rec = TableRecognitionPipelineV2(device=self._paddle_device)
                    self._log.info('TableRecognition ready')
            x1,y1,x2,y2 = map(int, layout.bbox)
            timg = img[y1:y2, x1:x2]
            if timg.size == 0: continue
            try:
                out = self._table_rec.predict(timg)
                if not out:
                    tables.append(Table(list(layout.bbox), '', layout.score))
                    continue
                html = ''.join(t.get('pred_html','') for t in out[0].get('table_res_list',[]) if t.get('pred_html'))
                tables.append(Table(list(layout.bbox), html, layout.score))
            except Exception as e:
                self._log.warning(f'Table OCR failed: {e}')
                tables.append(Table(list(layout.bbox), '', layout.score))
        return tables

    def _build_page(self, doc_name, elements, tables):
        header, footer, content = [], [], []
        page_number = ''
        elems = sorted(elements, key=lambda e: min(p[1] for p in e.box))
        for elem in elems:
            if elem.label == 'header': header.append(elem.text)
            elif elem.label == 'footer': footer.append(elem.text)
            elif elem.label == 'number': page_number = elem.text
            elif elem.label == 'page_title': content.insert(0, elem.text)
            elif elem.label == 'table': pass
            else: content.append(elem.text)
        return Page(doc_name, _parse_page_number(page_number),
                  ' '.join(header), content, ' '.join(footer), tables)

print('PaddleVietOCR OK (multi-GPU, thread-safe)')


In [ ]:
# === Output & Pipeline Orchestration (Multi-GPU: round-robin page distribution) ===
import threading
import queue

class YAMLOutput:
    def __init__(self, fs): self.fs = fs; self._lock = threading.Lock()
    def add(self, page):
        data = yaml.dump(page.to_dict(), allow_unicode=True, sort_keys=False)
        with self._lock:
            self.fs.put(FileInput(f'yaml/{page.doc_name}/{page.number}.yaml', data.encode('utf-8')))
    def scan_existing_pages(self, dest):
        result = {}
        for item in dest.ls():
            n = unicodedata.normalize('NFC', item)
            if n.startswith('yaml/') and n.endswith('.yaml'):
                parts = n.split('/')
                if len(parts) >= 3:
                    try:
                        result.setdefault(parts[1], set()).add(int(parts[-1].replace('.yaml','')))
                    except: pass
        return result

def _need_continue(files, dest, output):
    existing = output.scan_existing_pages(dest)
    done = set(existing.keys())
    need = []
    for f in files:
        book = unicodedata.normalize('NFC', f)
        if book not in done:
            need.append({'name': f, 'status': 'begin', 'ocr_checkpoint': -1})
        else:
            pages = existing.get(book, set())
            last = max(pages) if pages else -1
            need.append({'name': f, 'status': 'incomplete', 'ocr_checkpoint': last})
    return need

def _image_to_png_bytes(img_data):
    if isinstance(img_data, np.ndarray):
        if len(img_data.shape) == 3:
            mode = 'RGB' if img_data.shape[2] == 3 else 'RGBA'
            pil = PILImage.fromarray(img_data, mode=mode)
        else:
            pil = PILImage.fromarray(img_data, mode='L')
    else: pil = img_data
    buf = BytesIO(); pil.save(buf, format='PNG'); buf.seek(0)
    return buf.getvalue()

def _load_existing_images(dest, book_name, start_page, end_page):
    log = get_logger('Pipeline')
    images = []
    book = unicodedata.normalize('NFC', book_name)
    prefix = f'images/{book}/'
    files = []
    for item in dest.ls():
        n = unicodedata.normalize('NFC', item)
        if n.startswith(prefix) and n.endswith('.png'):
            try:
                num = int(n.split('/')[-1].replace('.png',''))
                files.append((num, n))
            except: pass
    files.sort(key=lambda x: x[0])
    for num, path in files:
        if start_page is not None and num < start_page: continue
        if end_page is not None and num > end_page: continue
        try:
            img_data = dest.read(path)
            images.append(Image(book_name, f'page_{num}', img_data))
        except Exception as e:
            log.warning(f'Failed to load {path}: {e}')
    log.info(f'Loaded {len(images)} existing images for {book[:40]}')
    return images

def _detect_gpu_count():
    """Detect number of available GPUs."""
    env = os.environ.get('BOOKTOTEXT_DEVICE','').strip().lower()
    if env == 'cpu': return 1
    try:
        import torch
        return torch.cuda.device_count() if torch.cuda.is_available() else 1
    except:
        return 1

# ---------- OCR Worker (runs in separate thread, owns one GPU) ----------
def _ocr_worker(ocr_queue, dest, ocr, output, stats, worker_id=0):
    """Consumer: picks up (file_info, images, indices) from queue, runs OCR on its GPU."""
    log = get_logger(f'OCR-Worker-{worker_id}')
    while True:
        item = ocr_queue.get()
        if item is None:  # Poison pill
            ocr_queue.task_done()
            break

        file_info, images, indices = item
        book = file_info['name'][:50]
        total = len(images)

        if total == 0:
            log.info(f'[GPU{worker_id}] No pages to OCR for {book}')
            ocr_queue.task_done()
            continue

        # indices can be a list (round-robin) or int (legacy start_idx)
        if isinstance(indices, int):
            page_indices = list(range(indices, indices + total))
        else:
            page_indices = indices

        log.info(f'[GPU{worker_id}] Starting {book} ({total} pages)')
        t_book = time.time()

        with tqdm(range(total), desc=f'GPU{worker_id} {book[:30]}', unit='pg') as pbar:
            for j in pbar:
                i = page_indices[j]
                t_page = time.time()
                image = images[j]

                # Save image to disk
                img_data = _image_to_png_bytes(image.get_data())
                dest.put(FileInput(f'images/{file_info["name"]}/{i}.png', img_data))

                # OCR on GPU
                page = ocr.ocr(image)
                if page:
                    if page.number == 0: page.number = i
                    output.add(page)

                elapsed = time.time() - t_page
                pbar.set_postfix(t=f'{elapsed:.1f}s', pg=i)

        elapsed_book = time.time() - t_book
        # stats dict is shared; CPython dict ops on simple keys are atomic enough for counters
        stats['books_done'] = stats.get('books_done', 0) + 1
        stats['pages_done'] = stats.get('pages_done', 0) + total
        log.info(f'[GPU{worker_id}] Done {book} in {elapsed_book:.1f}s ({total} pages, {elapsed_book/max(total,1):.2f}s/pg)')
        ocr_queue.task_done()

# ---------- Producer: PDF->Image (CPU-bound) ----------
def _convert_book(file_info, src, dest, converter, start_page, end_page):
    """Reads PDF and converts to images. Returns (images, ocr_start_idx)."""
    log = get_logger('Converter')
    book = file_info['name'][:50]

    log.info(f'[READ] {book}...')
    t0 = time.time()
    file_bytes = src.read(file_info['name'])
    log.info(f'[READ] Done ({len(file_bytes)/1024/1024:.1f}MB, {time.time()-t0:.1f}s)')

    pdf = PDF(file_info['name'], file_bytes)
    effective_start = start_page if start_page is not None else 0
    effective_end = end_page

    log.info(f'[CONVERT] {book}...')
    t0 = time.time()
    images = converter.convert(pdf, effective_start, effective_end, dest)
    log.info(f'[CONVERT] Done: {len(images)} images ({time.time()-t0:.1f}s)')

    if len(images) == 0:
        log.info(f'[LOAD] Loading existing images for {book}...')
        t0 = time.time()
        images = _load_existing_images(dest, file_info['name'], effective_start, effective_end)
        log.info(f'[LOAD] Done: {len(images)} images ({time.time()-t0:.1f}s)')

    # Determine OCR start based on checkpoint
    ocr_start = 0 if file_info['status'] == 'begin' else file_info.get('ocr_checkpoint', -1) + 1
    start = max(ocr_start, effective_start)
    end_val = effective_end if effective_end is not None else len(images)
    end_idx = min(end_val + 1, len(images))

    # Slice images to only what needs OCR
    if start > 0 or end_idx < len(images):
        images = images[start:end_idx]

    return images, start

def run_pipeline(src, dest, converter, ocr_or_device_ids, output, start_page=None, end_page=None, pdf_list=None):
    log = get_logger('Pipeline')
    log.info('Listing input PDFs...')
    if pdf_list is not None:
        pdfs = pdf_list
    else:
        pdfs = src.ls()
    log.info(f'Found {len(pdfs)} PDFs')

    to_convert = _need_continue(pdfs, dest, output)
    if not to_convert:
        log.info('Nothing to process')
        return

    # Determine GPU count and create OCR instances
    num_gpus = _detect_gpu_count()
    log.info(f'Detected {num_gpus} GPU(s)')

    if isinstance(ocr_or_device_ids, PaddleVietOCR):
        # Backward compat: single pre-built OCR instance
        ocr_instances = [ocr_or_device_ids]
        num_gpus = 1
    else:
        # None -> auto-detect all GPUs; list[int] -> use those device ids
        device_ids = list(range(num_gpus)) if ocr_or_device_ids is None else list(ocr_or_device_ids)
        num_gpus = len(device_ids)
        ocr_instances = []
        for gpu_id in device_ids:
            log.info(f'Initializing OCR engine on GPU {gpu_id}...')
            ocr_instances.append(PaddleVietOCR(device_id=gpu_id))

    print(f'{"="*60}')
    print(f'{len(to_convert)} books to process ({num_gpus} GPU workers, round-robin)')
    print(f'{"="*60}')

    # One queue per worker for round-robin distribution
    ocr_queues = [queue.Queue(maxsize=30) for _ in range(num_gpus)]
    stats = {}

    # Start OCR worker threads (one per GPU)
    workers = []
    for gpu_id in range(num_gpus):
        w = threading.Thread(
            target=_ocr_worker,
            args=(ocr_queues[gpu_id], dest, ocr_instances[gpu_id], output, stats, gpu_id),
            daemon=True
        )
        w.start()
        workers.append(w)

    # Main thread: convert books and distribute pages round-robin
    t_total = time.time()
    for idx, f in enumerate(to_convert, 1):
        print(f'{"_"*60}')
        print(f'[{idx}/{len(to_convert)}] CONVERT: {f["name"][:60]}')
        print(f'  Status: {f["status"]}, checkpoint: {f.get("ocr_checkpoint",-1)}')
        print(f'{"_"*60}')

        t_book = time.time()
        images, ocr_start = _convert_book(f, src, dest, converter, start_page, end_page)
        print(f'  Convert done in {time.time()-t_book:.1f}s, {len(images)} pages -> distributing to {num_gpus} GPU(s)')

        if len(images) > 0:
            if num_gpus == 1:
                ocr_queues[0].put((f, images, ocr_start))
            else:
                # Round-robin: split pages by index
                gpu_pages = [[] for _ in range(num_gpus)]
                gpu_indices = [[] for _ in range(num_gpus)]
                for i, img in enumerate(images):
                    gpu_id = i % num_gpus
                    gpu_pages[gpu_id].append(img)
                    gpu_indices[gpu_id].append(ocr_start + i)

                for gpu_id in range(num_gpus):
                    if gpu_pages[gpu_id]:
                        ocr_queues[gpu_id].put((f, gpu_pages[gpu_id], gpu_indices[gpu_id]))
        else:
            print(f'  No pages to OCR, skipping')

    # Send poison pills and wait for all workers
    for q in ocr_queues:
        q.put(None)
    for w in workers:
        w.join()

    elapsed = time.time() - t_total
    print(f'{"="*60}')
    print(f'Pipeline complete: {stats.get("books_done",0)} books, {stats.get("pages_done",0)} pages')
    print(f'Total time: {elapsed:.1f}s ({num_gpus} GPU(s))')
    print(f'{"="*60}')

    # Print combined stats from all OCR instances
    for gpu_id, ocr in enumerate(ocr_instances):
        print(f'--- GPU {gpu_id} Stats ---')
        ocr.print_stats()

print('Output & Pipeline OK (dual-GPU: round-robin page distribution)')


## 7. Run OCR Pipeline

In [ ]:
# === PDF & Page Selection ===
# Leave SELECTED_PDFS empty [] to process ALL PDFs in input folder.
# Otherwise, list specific filenames to process:
SELECTED_PDFS = [
    # 'Lịch sử Việt Nam tập 01.pdf',
    # 'Lịch sử Việt Nam tập 02.pdf',
]

# Page range (applies to ALL selected PDFs). None = all pages.
START_PAGE = None  # e.g. 0 for first page
END_PAGE = None    # e.g. 99 for first 100 pages

# --- Show available PDFs ---
src = FileSystem('/kaggle/working/input')
all_pdfs = [f for f in src.ls() if f.endswith('.pdf')]
print(f'Available PDFs ({len(all_pdfs)}):')
for pdf in sorted(all_pdfs):
    print(f'  - {pdf}')

pdf_list = [f for f in all_pdfs if f in SELECTED_PDFS] if SELECTED_PDFS else all_pdfs
if SELECTED_PDFS:
    missing = set(SELECTED_PDFS) - set(all_pdfs)
    if missing:
        print(f'\nWARNING: not found: {missing}')
print(f'\nWill process {len(pdf_list)} PDF(s):')
for pdf in sorted(pdf_list):
    print(f'  → {pdf}')


In [ ]:
_MODEL_CACHE.clear()

dest = FileSystem('/kaggle/working/output')
converter = PyPDFConverter(dpi=300)
output = YAMLOutput(dest)

# Show device info
import torch, paddle
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
print(f'paddle.is_compiled_with_cuda(): {paddle.is_compiled_with_cuda()}')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f'GPU count: {n_gpu}')
    for i in range(n_gpu):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

t0 = time.time()
run_pipeline(src, dest, converter, None, output,
             start_page=START_PAGE, end_page=END_PAGE, pdf_list=pdf_list)
print(f'Total time: {time.time()-t0:.1f}s ({(time.time()-t0)/60:.1f}m)')

yaml_files = glob.glob('/kaggle/working/output/yaml/**/*.yaml', recursive=True)
print(f'Generated {len(yaml_files)} YAML files')


## 8. View Sample Output

In [ ]:
yaml_files = sorted(glob.glob('/kaggle/working/output/yaml/**/*.yaml', recursive=True))
if yaml_files:
    with open(yaml_files[0]) as f:
        sample = yaml.safe_load(f)
    print(yaml.dump(sample, allow_unicode=True, sort_keys=False))
else:
    print('No YAML output found yet.')

## 9. Package Results

In [ ]:
!cd /kaggle/working && tar -czf /kaggle/working/ booktotext_output.tar.gz output/
!ls -lh /kaggle/working/booktotext_output.tar.gz

## 10. Upload to Cloudflare R2

In [ ]:
import boto3, os
from datetime import datetime

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


R2_ENDPOINT = 'https://9f3da53a23853c4184c9d1f305b39ad2.r2.cloudflarestorage.com'
R2_BUCKET = 'public-250201077'
R2_ACCESS_KEY = 'f97e00524453bc94b37853bce5f72e90'
R2_SECRET_KEY = user_secrets.get_secret("CLOUDFLARE_API_KEY")

s3 = boto3.client('s3',
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
    region_name='auto'
)

tar_path = '/kaggle/working/booktotext_output.tar.gz'
if os.path.exists(tar_path):
    size_mb = os.path.getsize(tar_path) / 1024 / 1024
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    key = f'booktotext/booktotext_output_{timestamp}.tar.gz'
    
    print(f'Uploading {tar_path} ({size_mb:.1f} MB) to R2...')
    s3.upload_file(tar_path, R2_BUCKET, key)
    print(f'Done! Uploaded to: s3://{R2_BUCKET}/{key}')
else:
    print('No tar.gz found. Run cell 29 first to package output.')
